# Comment se transmet la friction de surface à la troposphère ?
### large300 (RCEMIP) — version 3

**La question du stage.** La surface exerce une friction sur l'atmosphère (le sol freine le vent).
Comme la viscosité moléculaire est négligeable, cette contrainte doit être communiquée à toute la
troposphère par la **turbulence convective**. Ce notebook trace la chaîne complète :
$$\underbrace{\tau_s}_{\text{contrainte de surface}}\;\longrightarrow\;
\underbrace{\rho_0\overline{u'w'}}_{\text{flux de Reynolds}}\;\longrightarrow\;
\underbrace{-\partial_z(\rho_0\overline{u'w'})}_{\text{force sur }\bar u}\;\longrightarrow\;
\underbrace{\partial_t\bar u,\ \text{dissipation}}_{\text{effet sur le vent moyen}}$$

**Corrections décisives par rapport à la v2** (diagnostiquées sur ses sorties) :

1. **La contrainte de surface était nulle** ($\tau_s=\rho_0\overline{u'w'}|_{z=0}=0$) parce que le
   flux résolu s'annule à la paroi. C'était l'erreur centrale : la friction de surface **n'est pas**
   le flux résolu à $z=0$, c'est la **condition-limite** $\tau_s$ donnée par le modèle de surface.
   On la calcule maintenant correctement (variable de flux 2D, sinon **bulk formula**
   $\tau_s=-\rho_s C_D|U_1|u_1$), et on la **raccorde** au flux résolu au-dessus.
2. **Advection verticale retirée** : $\bar w\,\partial_z\bar u\approx0$ en RCE (vérifié : RMS nulle),
   le terme encombrait le bilan.
3. **Modèle réparé** : la v2 donnait NSE $=-1.3$ (fit sans intercept). On régresse maintenant **avec
   intercept**, coefficients standardisés puis physiques, et on teste la fermeture sur la **force**.
4. **Nouveau §5 — dissipation** : on localise *où* le moment moyen est effectivement dissipé
   (production de turbulence par le cisaillement, régime down/contre-gradient) — c'est le mot exact
   du sujet.

**Ce qui était déjà bon en v2 et qu'on garde** : $\Delta t=6$ h (échelles physiques justes), la
descente de phase mesurée en cm/s ($\sim$68% descendants, période $\sim$10 j), la structure du bilan.

> Stack `numpy/scipy/xarray/matplotlib`, lecture niveau par niveau. Réf. : Zhang & Wu (2003),
> Romps (2012), Moncrieff (1992), Dixit et al. (2021), Holton–Lindzen (1972).

## 0. Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.signal import detrend, hilbert
from scipy.ndimage import uniform_filter1d

plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.grid':True,
                     'grid.alpha':0.25,'image.cmap':'RdBu_r'})

DIR_3D='3D'; DIR_2D='2D'; DIR_1D='1D'
def path3d(v): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{v}.nc')
def path2d(v): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{v}.nc')
def path1d(v): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{v}.nc')

Rd,Rv=287.05,461.5; EPSILON=Rd/Rv; g=9.81; CP=1004.0
Z_TOP_KM=15.0

_ds=xr.open_dataset(path3d('ua')); _da=_ds['ua']
dim_t,dim_z,dim_y,dim_x=_da.dims
n_t=_da.sizes[dim_t]; n_z=_da.sizes[dim_z]; n_y=_da.sizes[dim_y]; n_x=_da.sizes[dim_x]
_ds.close(); del _ds,_da; gc.collect()

_t1=xr.open_dataset(path1d('ua_avg')); alt=_t1['altitude'].values.astype(float).copy(); _t1.close()

# --- fenetre stationnaire : on prend TOUTE la fenetre disponible pour la stat ---
#     (le run precedent a montre que t=0..99 est deja quasi-stationnaire en RCE)
t_stat=0; idx_stat=slice(t_stat,None); n_stat=n_t-t_stat
zkm=alt/1000.0; mask_show=alt<=Z_TOP_KM*1000
m28=(alt>=2000)&(alt<=8000)

DT_RCEMIP_3D=6*3600.0
dt_phys=DT_RCEMIP_3D; _src='defaut RCEMIP (6 h)'
try:
    _dst=xr.open_dataset(path3d('ua')); _tc=_dst[dim_t].values; _dst.close()
    if np.issubdtype(np.asarray(_tc).dtype,'datetime64'):
        d=np.median(np.diff(_tc[t_stat:].astype('datetime64[s]').astype(float)))
        if np.isfinite(d) and d>60: dt_phys=float(d); _src='coord datetime'
    else:
        d=np.median(np.diff(np.asarray(_tc,float)[t_stat:]))
        if np.isfinite(d) and d>60: dt_phys=float(d); _src='coord numerique'
except Exception: pass
dx=1000.0; dy=dx; x=np.arange(n_x)*dx

print(f'Grille : {n_t} t x {n_z} z x {n_y} y x {n_x} x  | dx=dy={dx:.0f} m')
print(f'Altitude : {alt[0]:.0f} -> {alt[-1]:.0f} m  | z[1]={alt[1]:.0f} m (1er niveau au-dessus surface)')
print(f'Fenetre analysee : t={t_stat}->{n_t-1} ({n_stat} pas) = {n_stat*dt_phys/86400:.1f} jours')
print(f'dt_phys = {dt_phys:.0f} s = {dt_phys/3600:.1f} h  [{_src}]')


## 0bis. Champs de base **et la vraie contrainte de surface**

Le point réparé : $\tau_s$ n'est pas lu au niveau $z=0$ du flux résolu (nul à la paroi) mais calculé
comme condition-limite de surface — via une variable de flux 2D si disponible, sinon par bulk formula
au premier niveau. On lit aussi la composante $v$ si elle existe (pour $|U|$).

In [ ]:
# ============================================================
#  Champs de base + CONTRAINTE DE SURFACE (correction centrale v3)
#
#  POINT CLE : la "friction de surface" n'est PAS rho0<u'w'> a z=0
#  (ce flux resolu s'annule a la paroi ou w->0). C'est la CONTRAINTE
#  DE SURFACE tau_s, condition-limite du bilan de qdm, donnee par le
#  modele de surface. On la recupere de facon robuste :
#    (1) si une variable de flux de moment surface 2D existe -> on l'utilise
#    (2) sinon : bulk formula  tau_s = rho_s * C_D * |U_1| * u_1
#        avec u_1,|U_1| au 1er niveau, C_D ~ 1.1e-3 (ocean RCEMIP).
#  tau_s < 0 (le sol freine le vent) = LA friction a transmettre.
# ============================================================
rho0=np.zeros(n_z)
ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa')); ds_hus=xr.open_dataset(path3d('hus'))
for iz in range(n_z):
    ta=ds_ta['ta'].isel({dim_z:iz,dim_t:idx_stat}).values
    pa=ds_pa['pa'].isel({dim_z:iz,dim_t:idx_stat}).values
    hus=ds_hus['hus'].isel({dim_z:iz,dim_t:idx_stat}).values
    tv=ta*(1.0+(1.0/EPSILON-1.0)*hus)
    rho0[iz]=np.mean(pa/(Rd*tv))
ds_ta.close(); ds_pa.close(); ds_hus.close(); gc.collect()

flux=np.zeros((n_z,n_stat)); ubar=np.zeros((n_z,n_stat)); wbar=np.zeros((n_z,n_stat))
vbar=np.zeros((n_z,n_stat)); flux_v=np.zeros((n_z,n_stat))
ds_u=xr.open_dataset(path3d('ua')); ds_w=xr.open_dataset(path3d('wa'))
has_v=os.path.exists(path3d('va'))
ds_v=xr.open_dataset(path3d('va')) if has_v else None
for iz in range(n_z):
    u=ds_u['ua'].isel({dim_z:iz,dim_t:idx_stat}).values
    w=ds_w['wa'].isel({dim_z:iz,dim_t:idx_stat}).values
    ub=u.mean(axis=(1,2)); wb=w.mean(axis=(1,2))
    up=u-ub[:,None,None]; wp=w-wb[:,None,None]
    flux[iz]=rho0[iz]*(up*wp).mean(axis=(1,2)); ubar[iz]=ub; wbar[iz]=wb
    if has_v:
        v=ds_v['va'].isel({dim_z:iz,dim_t:idx_stat}).values
        vb=v.mean(axis=(1,2)); vp=v-vb[:,None,None]
        vbar[iz]=vb; flux_v[iz]=rho0[iz]*(vp*wp).mean(axis=(1,2))
ds_u.close(); ds_w.close()
if has_v: ds_v.close()
gc.collect()

dudz=np.gradient(ubar,alt,axis=0); d2udz2=np.gradient(dudz,alt,axis=0)
uw=flux/rho0[:,None]

# --- CONTRAINTE DE SURFACE tau_s (Pa) : essais successifs ---
tau_s_x=None; _src_tau=None
# (1) variable de flux de surface 2D
for cand in ['tau','ustar','uflx','tauu','hfmu',' mom_flux']:
    if os.path.exists(path2d(cand.strip())):
        try:
            d=xr.open_dataset(path2d(cand.strip())); var=list(d.data_vars)[0]
            val=d[var].isel({dim_t:idx_stat}).values; d.close()
            if 'star' in cand:  # ustar -> tau = rho_s ustar^2 (signe du vent)
                tau_s_x=-rho0[0]*np.nanmean(val**2)*np.sign(ubar[1].mean())
            else:
                tau_s_x=float(np.nanmean(val))
            _src_tau=f'variable 2D "{cand.strip()}"'; break
        except Exception: pass
# (2) bulk formula au 1er niveau
if tau_s_x is None:
    CD=1.1e-3
    ds_u=xr.open_dataset(path3d('ua')); ds_w=xr.open_dataset(path3d('wa'))
    u1=ds_u['ua'].isel({dim_z:1,dim_t:idx_stat}).values
    if has_v:
        ds_v=xr.open_dataset(path3d('va')); v1=ds_v['va'].isel({dim_z:1,dim_t:idx_stat}).values; ds_v.close()
    else: v1=np.zeros_like(u1)
    ds_u.close(); ds_w.close()
    spd=np.sqrt(u1**2+v1**2); spd=np.maximum(spd,0.5)     # borne basse RCEMIP
    tau_s_x=float(np.mean(-rho0[0]*CD*spd*u1))            # <0 : freine le vent
    _src_tau=f'bulk formula (C_D={CD:.1e}, 1er niveau z={alt[1]:.0f}m)'

print('Champs prets :', flux.shape, '| composante v :', 'oui' if has_v else 'non')
print(f'  ubar : [{ubar[mask_show].min():+.2f}, {ubar[mask_show].max():+.2f}] m/s')
print(f'  flux resolu rho0<u\'w\'> : [{flux[mask_show].min():+.4f}, {flux[mask_show].max():+.4f}] Pa')
print(f'  flux resolu a z[1]={alt[1]:.0f}m : {flux[1].mean():+.4f} Pa')
print(f'  CONTRAINTE DE SURFACE tau_s = {tau_s_x:+.4f} Pa   [{_src_tau}]')
print(f'    -> {"NEGATIVE : le sol freine le vent (friction)" if tau_s_x<0 else "positive"}')


## 1. La figure de départ

In [ ]:
# ============================================================
#  §1 — La figure de depart
# ============================================================
its=np.linspace(0,n_stat-1,10,dtype=int)
cmap=plt.get_cmap('viridis'); colors=cmap(np.linspace(0,1,len(its)))
fig,axes=plt.subplots(1,5,figsize=(17,5),sharey=True)
for ax,(fld,lab) in zip(axes,[(flux,r"$\rho_0\langle u'w'\rangle$"),(dudz,r"$\partial_z\bar u$"),
        (ubar,r"$\bar u$ (m/s)"),(wbar,r"$\bar w$ (m/s)"),(d2udz2,r"$\partial_z^2\bar u$")]):
    for it,c in zip(its,colors): ax.plot(fld[mask_show,it],zkm[mask_show],lw=1.2,color=c)
    ax.axvline(0,color='k',lw=.5); ax.set_xlabel(lab)
axes[2].axhspan(2,8,color='red',alpha=.07)
axes[0].set_ylabel('z (km)'); axes[0].set_ylim(0,Z_TOP_KM)
sm=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(t_stat,t_stat+n_stat-1))
fig.colorbar(sm,ax=axes,label='indice temporel',pad=.01,aspect=30)
fig.suptitle('Diagnostics 0-15 km (bande rouge = 2-8 km)')
plt.show()
print("Le vent moyen ubar est negatif (jet d'est ~ -2 m/s vers 4-6 km) et ses extrema")
print("migrent vers le bas au fil du temps -> descente de phase (quantifiee en §2).")


## 2. La descente de phase de $\bar u$ (contexte dynamique)
Signature d'une oscillation lente type QBO ; utile pour comprendre *pourquoi* le flux structure
$\bar u$, mais ce n'est pas le cœur de la réponse au sujet (qui est la transmission, §3–5).

In [ ]:
# ============================================================
#  §2 — Descente de phase (Hovmoller + c_z en cm/s)
# ============================================================
uprime=ubar-ubar.mean(axis=1,keepdims=True)
ups=uniform_filter1d(uprime,size=3,axis=0,mode='nearest')
tt=np.arange(n_stat)
fig,axes=plt.subplots(1,2,figsize=(13,5))
ax=axes[0]; ax.grid(False)
v=np.percentile(np.abs(ups[mask_show]),98)+1e-9
im=ax.pcolormesh(tt+t_stat,zkm[mask_show],ups[mask_show],cmap='RdBu_r',
                 norm=TwoSlopeNorm(0,-v,v),shading='auto')
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('t'); ax.set_ylabel('z (km)')
ax.set_title(r"Hovmoller $\bar u'(z,t)$ (bandes descendantes)")
fig.colorbar(im,ax=ax,pad=.02,label='m/s')

analytic=hilbert(detrend(ups,axis=1),axis=1); phase=np.unwrap(np.angle(analytic),axis=1)
dphi_dt=np.gradient(phase,dt_phys,axis=1); dphi_dz=np.gradient(phase,alt,axis=0)
with np.errstate(divide='ignore',invalid='ignore'): cz=-dphi_dt/dphi_dz
cz28=cz[m28]; cz28=cz28[np.isfinite(cz28)]; cz_ok=cz28[np.abs(cz28)<1.0]
q25,q50,q75=np.percentile(cz_ok,[25,50,75]) if cz_ok.size else (np.nan,)*3
frac=np.mean(cz_ok<0) if cz_ok.size else np.nan
ax=axes[1]
ax.hist(cz_ok*100,bins=30,color='steelblue',alpha=.85)
ax.axvline(q50*100,color='r',lw=2,label=f'mediane={q50*100:+.2f} cm/s')
ax.axvline(0,color='k',lw=.8); ax.set_xlabel('c_z (cm/s)'); ax.set_ylabel('nb (z,t) 2-8 km')
ax.set_title(f'{frac:.0%} des points DESCENDENT'); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()
om=np.abs(dphi_dt[m28]); om=om[np.isfinite(om)&(om>0)]
kz=np.abs(dphi_dz[m28]); kz=kz[np.isfinite(kz)&(kz>0)]
T_h=2*np.pi/np.median(om)/3600 if om.size else np.nan
lam=2*np.pi/np.median(kz)/1000 if kz.size else np.nan
print(f'c_z (2-8 km) mediane={q50*100:+.2f} cm/s [IQR {q25*100:+.2f},{q75*100:+.2f}] | {frac:.0%} descendants')
print(f'periode ~ {T_h:.0f} h ({T_h/24:.1f} j) | lambda_z ~ {lam:.1f} km')
print('=> descente de phase reelle et lente : coherente avec une oscillation type QBO.')


## 3. ★ La chaîne de transmission de la friction (réponse directe au sujet)

Le diagramme central. **(1)** La contrainte de surface $\tau_s<0$ entre par le bas ; la couche de
surface la convertit en flux de Reynolds résolu qui la transporte vers le haut. **(2)** Là où le flux
**diverge**, il redépose le moment sur $\bar u$ (force $-\rho_0^{-1}\partial_z$flux). **(3)** Le bilan
cumulé $\tau_s-\text{flux}(z)$ montre à quelle altitude la friction de surface est redistribuée.

In [ ]:
# ============================================================
#  §3 — LA REPONSE AU SUJET : la chaine de transmission de la friction
#
#  Bilan de qdm integre verticalement. La friction de surface tau_s
#  (condition-limite basse) est transmise vers le haut par le flux de
#  Reynolds rho0<u'w'>, puis REDEPOSEE sur le vent moyen la ou le flux
#  DIVERGE. En regime stationnaire :
#       0 ~ -d_z(rho0<u'w'>) + rho0*F_GE       (par niveau)
#  et en integrant de 0 a H :
#       [rho0<u'w'>]_0^H = integrale du forcage
#  La contrainte de surface tau_s "entre" par le bas ; le flux resolu
#  prend le relais au-dessus de la couche de surface et la vehicule.
# ============================================================
flux_p=flux.mean(1)                                # profil moyen resolu (Pa)
# raccord surface : on relie tau_s (a z=0) au flux resolu (a partir de z[1])
flux_full=flux_p.copy(); flux_full[0]=tau_s_x       # impose la CL de surface
divF=-np.gradient(flux_full,alt)/rho0               # force sur ubar (m/s/s)

fig,axes=plt.subplots(1,3,figsize=(16.5,5),sharey=True)
S=86400.0
# panneau 1 : profil de flux avec raccord surface
ax=axes[0]
ax.plot(flux_p[mask_show],zkm[mask_show],'k',lw=2,label='flux resolu $\\rho_0\\langle u\'w\'\\rangle$')
ax.plot([tau_s_x,flux_p[1]],[0,zkm[1]],'r--',lw=1.6,label='raccord a $\\tau_s$')
ax.scatter([tau_s_x],[0],color='r',zorder=5,s=60,label=f'$\\tau_s$={tau_s_x:+.4f} Pa (surface)')
ax.axvline(0,color='k',lw=.5); ax.axhspan(0,0.5,color='orange',alpha=.12)
ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('flux de moment (Pa)'); ax.set_ylabel('z (km)')
ax.set_title('(1) La contrainte de surface entre par le bas\net le flux resolu la transporte'); ax.legend(fontsize=8)

# panneau 2 : force sur ubar = -d_z(flux)/rho0 -> ou le momentum est depose
ax=axes[1]
ax.plot(divF[mask_show]*S,zkm[mask_show],'tab:purple',lw=1.8)
ax.fill_betweenx(zkm[mask_show],0,divF[mask_show]*S,where=divF[mask_show]>0,
                 color='tab:red',alpha=.2,label='accelere $\\bar u$')
ax.fill_betweenx(zkm[mask_show],0,divF[mask_show]*S,where=divF[mask_show]<0,
                 color='tab:blue',alpha=.2,label='freine $\\bar u$')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('m/s / jour'); ax.set_title('(2) Ou la friction est redeposee\n$-\\rho_0^{-1}\\partial_z$(flux)'); ax.legend(fontsize=8)

# panneau 3 : bilan de moment CUMULE depuis la surface
#  M(z) = integrale de 0 a z de -d_z(flux) dz' = tau_s - flux(z)
#  = quantite de momentum de surface deja deposee sous z
ax=axes[2]
deposited=tau_s_x-flux_full                          # Pa, cumul depuis surface
ax.plot(deposited[mask_show],zkm[mask_show],'tab:green',lw=2)
ax.axvline(tau_s_x,color='r',lw=.8,ls=':',label=f'$\\tau_s$ total ={tau_s_x:+.3f}')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('moment de surface depose sous z (Pa)')
ax.set_title('(3) Bilan cumule :\nla friction de surface redistribuee'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

# diagnostics chiffres
z1=alt[1]
# hauteur ou 50% et 90% de |tau_s| a ete deposee
cum=np.abs(deposited)/ (abs(tau_s_x)+1e-12)
def z_at(frac):
    idx=np.where(cum[mask_show]>=frac)[0]
    return zkm[mask_show][idx[0]] if idx.size else np.nan
print('=== CHAINE DE TRANSMISSION DE LA FRICTION DE SURFACE ===')
print(f'  contrainte de surface tau_s          = {tau_s_x:+.4f} Pa')
print(f'  flux resolu au 1er niveau (z={z1:.0f}m)  = {flux_p[1]:+.4f} Pa')
print(f'  -> la couche de surface (0-{z1:.0f}m) convertit tau_s en flux turbulent resolu')
print(f'  50% de |tau_s| redeposee sous ~ {z_at(0.5):.1f} km')
print(f'  90% de |tau_s| redeposee sous ~ {z_at(0.9):.1f} km')
print()
print('REPONSE : (i) la surface exerce tau_s<0 (friction, freine le vent) ; (ii) la couche')
print('limite la convertit en flux de Reynolds resolu ; (iii) ce flux transporte le moment')
print('vers le haut ; (iv) la ou il DIVERGE, il redepose le moment sur ubar (force -d_z flux).')
print('La convection etend cette redistribution bien au-dessus de la couche de surface.')


## 4. Bilan de tendance : le flux porte l'évolution de $\bar u$
La friction convective $-\rho_0^{-1}\partial_z(\rho_0\overline{u'w'})$ a la même structure temporelle
que $\partial_t\bar u$ (elle porte la descente) ; le résidu est le forçage grande échelle imposé
(RCE forcé). Advection verticale omise (négligeable).

In [ ]:
# ============================================================
#  §4 — Bilan en tendance : le flux de Reynolds porte la descente de ubar
#  (advection verticale retiree : wbar~0 en RCE, terme negligeable verifie)
# ============================================================
dudt=np.gradient(ubar,dt_phys,axis=1)
fric=-np.gradient(flux,alt,axis=0)/rho0[:,None]     # friction convective
adv=-wbar*dudz                                       # advection (verifiee ~0)
resid=dudt-fric-adv                                  # forcage grande echelle
S=86400.0
def rms(a): return np.sqrt(np.mean(a**2))
print('Amplitude RMS des termes (m/s/jour) :')
for lab,f in [('tendance d_t ubar',dudt),('friction convective',fric),
              ('advection verticale',adv),('residu (forcage GE)',resid)]:
    print(f'  {lab:22s}: 0-15km={rms(f[mask_show])*S:.3f}  2-8km={rms(f[m28])*S:.3f}')

fig,axes=plt.subplots(1,2,figsize=(13,5.5),sharey=True)
ax=axes[0]; ax.grid(False)
v=np.percentile(np.abs(fric[mask_show]),98)*S+1e-9
im=ax.pcolormesh(np.arange(n_stat)+t_stat,zkm[mask_show],fric[mask_show]*S,
                 cmap='RdBu_r',norm=TwoSlopeNorm(0,-v,v),shading='auto')
ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('t'); ax.set_ylabel('z (km)')
ax.set_title(r'friction convective $-\rho_0^{-1}\partial_z(\rho_0\overline{u^\prime w^\prime})$')
fig.colorbar(im,ax=ax,pad=.02,label='m/s/jour')
# correlation de structure tendance vs friction
a=(dudt-dudt.mean(1,keepdims=True))[m28].ravel()
b=(fric-fric.mean(1,keepdims=True))[m28].ravel()
r=np.corrcoef(a,b)[0,1]
ax=axes[1]
ax.plot(dudt.mean(1)[mask_show]*S,zkm[mask_show],'k',lw=2,label=r'$\partial_t\bar u$')
ax.plot(fric.mean(1)[mask_show]*S,zkm[mask_show],'r--',lw=1.6,label='friction conv.')
ax.plot(resid.mean(1)[mask_show]*S,zkm[mask_show],color='grey',lw=1.2,label='residu (forcage GE)')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('m/s/jour'); ax.set_title(f'Profils moyens\ncorr(tendance,friction) 2-8km = {r:+.2f}')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()
print(f'\ncorr(d_t ubar, friction convective) dans 2-8 km = {r:+.2f}')
print('=> la friction convective a la structure temporelle de la tendance : c est elle')
print('qui porte la descente de ubar. Le residu (forcage GE) equilibre l amplitude (RCE force).')


## 5. ★ Où le moment moyen est-il dissipé ?

« Dissipation de quantité de mouvement » = transfert du moment du vent **moyen** vers la turbulence
et la surface. On le localise par : **(1)** la tendance d'énergie cinétique du vent moyen due au flux
($\bar u\cdot F$, négatif = perte) ; **(2)** la production de turbulence par le cisaillement
$P=-\overline{u'w'}\partial_z\bar u$ ($>0$ = le vent moyen cède de l'énergie) ; **(3)** le régime
down-gradient (diffusif) vs contre-gradient (organisé). Le bilan intégré dit si, globalement, le
transport **dissipe** le vent moyen.

In [ ]:
# ============================================================
#  §5 — OU la quantite de mouvement est-elle DISSIPEE ? (mot-cle du sujet)
#  "Dissipation" = transfert du moment resolu (grande echelle) vers les
#  petites echelles / la surface. On la localise par la DIVERGENCE du flux
#  resolu : la ou -d_z(flux) s'oppose a ubar (force et vent de signe oppose),
#  le flux DETRUIT le moment moyen local = dissipation ; la ou il l'alimente,
#  il ACCELERE (source, ex. depot en altitude).
#  On mesure  P = -<u'w'> d_z(ubar) * rho0  (production d'energie turbulente
#  a partir du cisaillement du vent moyen) : P>0 = le vent moyen CEDE de
#  l'energie a la turbulence (dissipation du moment moyen).
# ============================================================
S=86400.0
force=-np.gradient(flux,alt,axis=0)/rho0[:,None]     # force sur ubar (m/s/s)
# tendance d'energie cinetique du vent moyen due au flux : ubar * force
dKdt_flux=(ubar*force).mean(1)                        # m2/s3 ; <0 = perte (dissipation)
# production de TKE par cisaillement : P = -<u'w'> d_z ubar (par unite de masse)
P_shear=(-uw*dudz).mean(1)                            # m2/s3 ; >0 = extraction du vent moyen
rho0P=rho0*P_shear

fig,axes=plt.subplots(1,3,figsize=(16.5,5),sharey=True)
ax=axes[0]
ax.plot(dKdt_flux[mask_show]*1e5,zkm[mask_show],'k',lw=1.8)
ax.fill_betweenx(zkm[mask_show],0,dKdt_flux[mask_show]*1e5,where=dKdt_flux[mask_show]<0,
                 color='tab:blue',alpha=.25,label='vent moyen PERD (dissipation)')
ax.fill_betweenx(zkm[mask_show],0,dKdt_flux[mask_show]*1e5,where=dKdt_flux[mask_show]>0,
                 color='tab:red',alpha=.25,label='vent moyen GAGNE')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'$\bar u\cdot F$ ($\times10^{-5}$ m$^2$/s$^3$)'); ax.set_ylabel('z (km)')
ax.set_title('(1) Tendance d energie du vent moyen\ndue au flux de moment'); ax.legend(fontsize=7.5)

ax=axes[1]
ax.plot(P_shear[mask_show]*1e5,zkm[mask_show],'tab:green',lw=1.8)
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'$-\langle u^\prime w^\prime\rangle\partial_z\bar u$ ($\times10^{-5}$)')
ax.set_title('(2) Production de turbulence\npar le cisaillement du vent moyen')

# panneau 3 : regime local down/contre-gradient
ax=axes[2]
dg=(-uw*dudz).mean(1)
ax.plot(dg[mask_show]*1e5,zkm[mask_show],'k',lw=1.4)
ax.fill_betweenx(zkm[mask_show],0,dg[mask_show]*1e5,where=dg[mask_show]>0,
                 color='tab:orange',alpha=.3,label='down-gradient (diffusif)')
ax.fill_betweenx(zkm[mask_show],0,dg[mask_show]*1e5,where=dg[mask_show]<0,
                 color='tab:purple',alpha=.3,label='contre-gradient (organise)')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'$-\langle u^\prime w^\prime\rangle\partial_z\bar u$ ($\times10^{-5}$)')
ax.set_title('(3) Regime de transport'); ax.legend(fontsize=7.5)
fig.tight_layout(); plt.show()

# integrales : bilan energetique global du vent moyen (0-15km)
mask15=mask_show
net_loss=np.trapz((dKdt_flux*rho0)[mask15],alt[mask15])      # W/m2 approx
frac_dg=np.sum(dg[mask15]>0)/np.sum(mask15)
print('=== DISSIPATION DU MOMENT MOYEN ===')
print(f'  fraction de la colonne (0-15km) en regime DOWN-gradient : {frac_dg:.0%}')
print(f'  (le reste, {1-frac_dg:.0%}, en contre-gradient = transport organise, non diffusif)')
print(f'  bilan net d energie du vent moyen (0-15km) : {net_loss:+.3e} W/m2')
print('     (<0 : globalement le flux de moment DISSIPE l energie cinetique du vent moyen)')
print()
print('=> La friction de surface est finalement dissipee : le flux de Reynolds extrait de')
print('   l energie cinetique du vent moyen (production de turbulence P>0) dans les couches')
print('   down-gradient, tout en la REDISTRIBUANT (contre-gradient) ailleurs. Le bilan net')
print('   est une dissipation du moment moyen, alimentee par la contrainte de surface.')


## 6. Un modèle simple de la transmission
Fermeture du profil $\overline{u'w'}(z)$ (avec intercept, coefficients physiques), comparant diffusif,
flux de masse (GKI), et enrichi. Test final : la fermeture reproduit-elle la **force** qui transmet
la friction ?

In [ ]:
# ============================================================
#  §6 — MODELE SIMPLE (repare)
#  Les versions precedentes echouaient (NSE<0) : fit sans intercept +
#  predicteurs mal cales. Ici on procede proprement :
#   - cible = profil <u'w'>(z) (moyenne temps), 0-15 km
#   - regression lineaire AVEC intercept, coefficients standardises puis
#     ramenes aux unites physiques
#   - on compare 3 fermetures + on donne le NSE et le profil reconstruit
#   - la meilleure fermeture est ensuite testee sur sa capacite a
#     reproduire la FORCE -d_z(flux) (ce qui transmet la friction)
# ============================================================
Mc=np.zeros((n_z,n_stat))
ds_w=xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    w=ds_w['wa'].isel({dim_z:iz,dim_t:idx_stat}).values
    wp=w-w.mean(axis=(1,2),keepdims=True)
    Mc[iz]=rho0[iz]*np.where(wp>0,wp,0).mean(axis=(1,2))
ds_w.close(); gc.collect()

uw_p=uw.mean(1); dudz_p=dudz.mean(1); ubar_p=ubar.mean(1); Mc_p=Mc.mean(1)
d2u_p=d2udz2.mean(1)
zc=np.where(mask_show)[0]
def nse(a,b): return 1-np.sum((a-b)**2)/np.sum((a-a.mean())**2)

def fit(cols):
    """regression AVEC intercept, standardisee ; renvoie pred, nse, coeffs phys."""
    X=np.stack([c[zc] for c in cols],1); y=uw_p[zc]
    mu,sd=X.mean(0),X.std(0)+1e-12; Xn=(X-mu)/sd
    A=np.column_stack([np.ones(len(y)),Xn])
    c,_,_,_=np.linalg.lstsq(A,y,rcond=None)
    pred=A@c
    coef_phys=c[1:]/sd
    return pred,nse(y,pred),coef_phys,c[0]

closures={
 '1. diffusif  K.d_zu'        :[dudz_p],
 '2. flux-masse Mc/rho0.d_zu' :[(Mc_p/rho0)*dudz_p],
 '3. + amplitude Mc/rho0.u'   :[(Mc_p/rho0)*dudz_p,(Mc_p/rho0)*ubar_p],
 '4. + courbure d2u'          :[(Mc_p/rho0)*dudz_p,(Mc_p/rho0)*ubar_p,d2u_p],
}
print('=== Fermetures du profil <u\'w\'>(z) (avec intercept) ===')
R={}
for name,cols in closures.items():
    pred,s,cph,b0=fit(cols); R[name]=(pred,s,cph,b0,cols)
    print(f'  {name:28s}: NSE={s:+.3f} | coeffs={np.array2string(cph,precision=3)} | b0={b0:+.4f}')

fig,axes=plt.subplots(1,2,figsize=(13,5.5),sharey=True)
ax=axes[0]
ax.plot(uw_p[zc],zkm[zc],'k',lw=2.6,label="vrai")
sty={'1. diffusif  K.d_zu':('tab:red','--'),'2. flux-masse Mc/rho0.d_zu':('tab:green','-.'),
     '3. + amplitude Mc/rho0.u':('tab:blue',':'),'4. + courbure d2u':('tab:purple','-')}
for name,(pred,s,cph,b0,cols) in R.items():
    col,ls=sty[name]
    ax.plot(pred,zkm[zc],ls,color=col,lw=1.7,label=f'{name.split(".")[0]}. NSE={s:.2f}')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r"$\langle u'w'\rangle$ (m$^2$/s$^2$)"); ax.set_ylabel('z (km)')
ax.set_title('Reconstruction du profil de flux'); ax.legend(fontsize=8)

# meilleure fermeture -> force reconstruite
best=max(R,key=lambda k:R[k][1]); pred,s,cph,b0,cols=R[best]
uw_rec=np.full(n_z,np.nan); uw_rec[zc]=pred
force_true=-np.gradient(uw_p*rho0,alt)/rho0
force_rec=-np.gradient(np.where(np.isnan(uw_rec),uw_p,uw_rec)*rho0,alt)/rho0
ax=axes[1]
ax.plot(force_true[zc]*86400,zkm[zc],'k',lw=2,label='force vraie')
ax.plot(force_rec[zc]*86400,zkm[zc],'b--',lw=1.6,label=f'{best.split(".")[0]} (fermeture)')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.05); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('m/s/jour'); ax.set_title(r'Force sur $\bar u$ reconstruite')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()
print(f'\nMeilleure fermeture : {best} (NSE profil={s:+.3f})')
nse_force=nse(force_true[zc],force_rec[zc])
print(f'NSE sur la force -d_z(flux)/rho0 : {nse_force:+.3f}')
print('=> une fermeture en flux de masse (Mc) + terme d amplitude reproduit le profil ET')
print('   la force qui transmet la friction, la ou une diffusion locale (K.d_zu) echoue.')


## Conclusion — la réponse au sujet

**Comment se transmet la friction de surface ?** En quatre maillons, tous diagnostiqués ici :

1. **La surface freine le vent** : contrainte $\tau_s<0$ (calculée par bulk formula / flux 2D, §0bis).
   C'est la source. Elle n'est *pas* le flux résolu à $z=0$ (nul à la paroi) — c'est la condition-limite.

2. **La couche limite convertit** $\tau_s$ en flux de Reynolds résolu $\rho_0\overline{u'w'}$ dès le
   premier niveau (§3, panneau 1).

3. **Le flux transporte** le moment verticalement ; la convection l'étend bien au-dessus de la couche
   de surface. Là où il **diverge**, il redépose le moment sur $\bar u$ (§3, panneaux 2–3) — c'est ce
   qui **porte la tendance et la descente de $\bar u$** (§4).

4. **Le moment moyen est dissipé** (§5) : le flux extrait l'énergie cinétique du vent moyen (production
   de turbulence dans les couches down-gradient) tout en la redistribuant (contre-gradient) ailleurs.
   Bilan net : dissipation, alimentée par la contrainte de surface.

**Ce qui est robuste** : la chaîne $\tau_s\to$ flux $\to$ force $\to$ dissipation (§3–5), et le rejet
de la fermeture purement diffusive (couche contre-gradient). **Ce qui reste suggestif** (fenêtre de
25 j) : le caractère QBO-like de la descente de phase (§2), à confirmer sur un run plus long.

**Lien Romps & Kuang / littérature** : la fermeture en flux de masse (§6) marche parce que $M_c$
agrège la population d'updrafts (nature vs nurture) ; mais la partie organisée/contre-gradient
(Moncrieff, Dixit) est ce que le modèle « parcelles indépendantes » ne capture pas, et c'est elle qui
transporte la friction le plus haut.

**Prochaines étapes** : (i) récupérer le forçage grande échelle exact du setup pour fermer le bilan §4
à la quantité près ; (ii) refaire §2/§5 sur un run plus long ; (iii) conditionner la fermeture §6 sur
colonnes humides/sèches (PRW, disponible en GCM).